# Citi Bike: tuned random forest

**Contribution owner: Tomislav Novosel.** Implementation, experiments and initial explanations were prepared with AI assistance on 9 October 2026. Human review and substantive changes should be recorded when they happen; see [the assistance record](../docs/AI_USE.md).

These are development experiments, not final-test results. The training/tuning dates end in June; July-September is development validation. October-December has not been scored.

The searches were executed with the helper scripts and their actual reports are saved. By default this notebook reviews those reports quickly. Set `CITIBIKE_RETRAIN=1` or use the runner's `--train` option to execute the search again. The XGBoost search needs a CUDA GPU; viewing results and running the saved app do not.


A random forest averages many trees rather than adding corrections sequentially. We test tree depth, minimum leaf size, feature sampling and a residual target.

In [12]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/citibike_forecast.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
from citibike_forecast import REPORTS, MODELS, development_data, chronological_cv
RUN_TRAINING = os.environ.get('CITIBIKE_RETRAIN') == '1'
def report(name):
    return json.loads((REPORTS / (name + '_run.json')).read_text(encoding='utf8'))
print('Mode:', 'execute training' if RUN_TRAINING else 'review actual saved experiments')
from citibike_additional import run_search
if RUN_TRAINING:
    run_search('random_forest')
run = report('random_forest_tuned')
display(pd.Series(run['search']))
grid = pd.read_csv(REPORTS / 'random_forest_grid.csv').sort_values('rank_test_score')
display(grid.filter(regex='param_|mean_test_score|rank_test_score').head(10))


Mode: review actual saved experiments


method                                                             GridSearchCV
configurations                                                               36
cv_fits                                                                     108
best_parameters               {'model__max_depth': 3, 'model__max_features':...
best_cv_mae                                                        14092.090794
elapsed_seconds                                                         5.82787
validation_used_for_search                                                False
dtype: object

,param_model__max_depth,param_model__max_features,param_model__min_samples_leaf,param_seasonal_residual,mean_test_score,rank_test_score
1,3.0,0.7,2,True,-14092.090794,1
9,3.0,1.0,5,True,-14248.154636,2
2,3.0,0.7,5,False,-14287.131669,3
3,3.0,0.7,5,True,-14291.731632,4
8,3.0,1.0,5,False,-14315.802615,5
14,6.0,0.7,5,False,-14330.229986,6
26,NaN,0.7,5,False,-14332.415230,7
32,NaN,1.0,5,False,-14376.038816,8
20,6.0,1.0,5,False,-14377.465530,9
5,3.0,0.7,10,True,-14541.421979,10


The 36 configurations use CPU workers. Random forest is close to XGBoost and has slightly better RMSE, but XGBoost wins on the primary MAE metric.

In [13]:
display(pd.Series(run['validation']))
display(pd.DataFrame([{'model': n, **report(n)['validation']} for n in ['random_forest_tuned','xgboost_tuned']]))
print('A small MAE difference does not establish that one family will always be better.')


A small MAE difference does not establish that one family will always be better.


mae     15007.238525
rmse    23229.036475
wape        0.124438
bias     2332.139422
days       92.000000
dtype: float64

,model,mae,rmse,wape,bias,days
0,random_forest_tuned,15007.238525,23229.036475,0.124438,2332.139422,92
1,xgboost_tuned,14692.652301,23426.393863,0.121829,3763.623936,92
